In [ ]:
# Snowflake Time Travel Demo
# This notebook demonstrates creating a table, modifying data,
# and using Time Travel to query historical states of the table.

In [ ]:
%%sql -r use_ctx
USE DATABASE SNOWFLAKE_ASSIGNMENT;
USE SCHEMA ASSIGNMENT_SCHEMA;

In [ ]:
%%sql -r create_orders
-- Create table with DATA_RETENTION_TIME_IN_DAYS for Time Travel
CREATE OR REPLACE TABLE ORDERS (
    ORDER_ID INT,
    CUSTOMER_NAME VARCHAR(100),
    PRODUCT VARCHAR(50),
    QUANTITY INT,
    ORDER_DATE DATE
) DATA_RETENTION_TIME_IN_DAYS = 1;

In [ ]:
%%sql -r insert_orders
-- Insert sample records
INSERT INTO ORDERS VALUES
(1, 'Alice', 'Laptop', 1, '2024-01-10'),
(2, 'Bob', 'Mouse', 3, '2024-01-11'),
(3, 'Charlie', 'Keyboard', 2, '2024-01-12'),
(4, 'Diana', 'Monitor', 1, '2024-01-13'),
(5, 'Eve', 'Headphones', 4, '2024-01-14');

In [ ]:
%%sql -r original_data
-- View original data
SELECT * FROM ORDERS ORDER BY ORDER_ID;

In [ ]:
%%sql -r set_ts
-- Capture the current timestamp BEFORE making changes
SET ts_before_changes = CURRENT_TIMESTAMP();

## Perform UPDATE and DELETE Operations

In [ ]:
%%sql -r update_result
-- UPDATE: Change quantity for Bob's order
UPDATE ORDERS
SET QUANTITY = 5
WHERE ORDER_ID = 2;

In [ ]:
%%sql -r delete_result
-- DELETE: Remove Diana's order
DELETE FROM ORDERS
WHERE ORDER_ID = 4;

In [ ]:
%%sql -r current_data
-- View current data (after UPDATE and DELETE)
SELECT * FROM ORDERS ORDER BY ORDER_ID;

## Time Travel - Query Historical Data

In [ ]:
%%sql -r tt_timestamp
-- Time Travel using TIMESTAMP: query the table BEFORE changes were made
SELECT * FROM ORDERS
    BEFORE (TIMESTAMP => $ts_before_changes)
ORDER BY ORDER_ID;

In [ ]:
%%sql -r tt_offset
-- Time Travel using OFFSET: query the table as it was 5 seconds ago
-- (Using a small offset to stay within the table's existence window)
SELECT * FROM ORDERS
    AT (OFFSET => -5)
ORDER BY ORDER_ID;

In [ ]:
%%sql -r compare_counts
-- Compare: Current row count vs Historical row count
SELECT 
    'Current' AS STATE, COUNT(*) AS ROW_COUNT FROM ORDERS
UNION ALL
SELECT 
    'Before Changes' AS STATE, COUNT(*) AS ROW_COUNT FROM ORDERS
    BEFORE (TIMESTAMP => $ts_before_changes);

In [ ]:
%%sql -r restore_result
-- Restore deleted record using Time Travel + INSERT
INSERT INTO ORDERS
    SELECT * FROM ORDERS
        BEFORE (TIMESTAMP => $ts_before_changes)
    WHERE ORDER_ID = 4;

In [ ]:
%%sql -r final_data
-- Verify the restored record
SELECT * FROM ORDERS ORDER BY ORDER_ID;